# 04 · Comparación de modelos ML — evaluación cuantitativa del cualificador**TFM Data Science e IA — Juan Alonso · Tutor: Julio Valero · Curso 2025-2026**## Objetivo de este notebookEste notebook evalúa cuantitativamente el sistema de cualificación de leads que se usa en la app Streamlit del proyecto. Compara tres enfoques sobre dos escenarios reales (los mismos que se demuestran en la app):1. **Baseline manual** — filtro por reglas duras, lo que haría un comercial sin ML.2. **Modelo 1 · Similitud coseno** — el que va en producción en la app.3. **Modelo 2 · XGBoost con Positive-Unlabeled** — clasificador supervisado como comparación.**Escenarios evaluados:**- **ICP premium** (Demo 1 en la app): Sagardi, Sandó, Dray Martina.- **ICP asiático** (Demo 2 en la app): Okashi Sanda, Hunan Restaurant, Tuk Tuk.## Nota importante sobre el alcance del vectorEl vector de features que utiliza el modelo se construye a partir de:- **Numéricas escaladas**: competencia_500m, competencia_1km, renta_media.- **Booleanas**: tiene_web, tiene_telefono, tiene_horario, tiene_terraza, ofrece_delivery, ofrece_takeaway, es_accesible.- **Categóricas one-hot**: cocina, epigrafe_oficial, barrio, ambiente, presencia_digital.**El embedding del resumen_llm NO forma parte de este vector en la versión actual del modelo** (`src/similitud.py`). Está identificado como línea futura de trabajo. La documentación del proyecto se ha corregido para reflejarlo con exactitud.## Nota importante sobre el rango del scoreEl score de similitud coseno está teóricamente en el rango **[-1, 1]**. En la práctica, sobre esta capa gold (donde el vector tiene muchas features no negativas), los valores observados caen en el rango **[0, 1]**. No se debe confundir con una probabilidad.## Hipótesis de partidaCon solo 3 seeds positivos, la teoría del *few-shot learning* predice que:- El modelo simple (similitud vectorial) será más estable que el complejo (XGBoost).- La métrica leave-one-out es especialmente exigente en este régimen — no captura necesariamente si el modelo aprende algo útil.Este notebook contrasta esas hipótesis con datos.

## 1 · Carga de la capa gold y librerías

In [ ]:
from __future__ import annotationsfrom pathlib import Pathimport sysimport warningswarnings.filterwarnings("ignore")import numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom sklearn.preprocessing import StandardScalerROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()sys.path.insert(0, str(ROOT))GOLD_PATH = ROOT / "data" / "gold" / "gold_restaurantes_madrid.parquet"gold = pd.read_parquet(GOLD_PATH)print(f"Capa gold cargada: {len(gold)} restaurantes × {len(gold.columns)} columnas")

## 2 · Vectorización — construcción del espacio de featuresVectorizamos cada restaurante con la MISMA lógica que usa el modelo en producción (`src/similitud.py`).**Excluimos por análisis de sesgo del LLM (baja varianza):** `posicionamiento_precio` (90% "medio") y `tipo_clientela` (84% "mixta"). Estas dos variables se mantienen en la app como campos descriptivos, pero se excluyen del vector.

In [ ]:
NUMERICAS = ["competencia_500m", "competencia_1km", "renta_media"]BOOLEANAS = ["tiene_web", "tiene_telefono", "tiene_horario", "tiene_terraza",             "ofrece_delivery", "ofrece_takeaway", "es_accesible"]CATEGORICAS = ["cocina", "epigrafe_oficial", "barrio", "ambiente", "presencia_digital"]def construir_matriz(df: pd.DataFrame):    df = df.copy()    numericas = [c for c in NUMERICAS if c in df.columns]    booleanas = [c for c in BOOLEANAS if c in df.columns]    categoricas = [c for c in CATEGORICAS if c in df.columns]    for c in booleanas:        df[c] = df[c].fillna(False).astype(int)    for c in categoricas:        df[c] = df[c].astype("string").fillna("desconocido")    df_num = df[numericas].copy()    for c in numericas:        df_num[c] = pd.to_numeric(df_num[c], errors="coerce")        df_num[c] = df_num[c].fillna(df_num[c].median())    df_cat = pd.get_dummies(df[categoricas], prefix=categoricas).astype(int)    X = pd.concat([df_num, df[booleanas], df_cat], axis=1)    feature_names = X.columns.tolist()    X_scaled = StandardScaler().fit_transform(X.values)    return X_scaled, feature_names, df["restaurante_id"].reset_index(drop=True)X, feature_names, ids = construir_matriz(gold)print(f"Matriz de features: {X.shape[0]} restaurantes × {X.shape[1]} features")print(f"(No incluye embedding del resumen_llm — pendiente como línea futura.)")

## 3 · Definición de los dos ICPsUsamos exactamente los mismos seeds que están configurados como presets en la app Streamlit.

In [ ]:
# ICP 1 — premium (Demo 1 en la app)SEEDS_PREMIUM = ["Sagardi", "Sandó", "Dray Martina"]# ICP 2 — asiático (Demo 2 en la app)SEEDS_ASIATICO = ["Okashi Sanda", "Hunan Restaurant", "Tuk Tuk"]def obtener_seeds(df: pd.DataFrame, nombres: list[str]) -> tuple[list[str], pd.DataFrame]:    seeds_df = df[df["nombre"].isin(nombres)].copy()    return seeds_df["restaurante_id"].tolist(), seeds_dfseeds_premium_ids, seeds_premium_df = obtener_seeds(gold, SEEDS_PREMIUM)seeds_asiatico_ids, seeds_asiatico_df = obtener_seeds(gold, SEEDS_ASIATICO)print("=== ICP PREMIUM ===")print(seeds_premium_df[["nombre", "barrio", "cocina", "posicionamiento_precio",                        "ambiente", "presencia_digital"]].to_string(index=False))print("\n=== ICP ASIÁTICO ===")print(seeds_asiatico_df[["nombre", "barrio", "cocina", "posicionamiento_precio",                         "ambiente", "presencia_digital"]].to_string(index=False))

## 4 · Modelo 1 · Similitud cosenoEl vector-ICP es el centroide (media) de los seeds. El score de cada restaurante es el coseno entre su vector y el centroide.

In [ ]:
def scores_coseno(X: np.ndarray, ids: pd.Series, seeds: list[str]) -> pd.Series:    mask = ids.isin(seeds).values    if mask.sum() == 0:        raise ValueError("Ningún seed encontrado.")    centroide = X[mask].mean(axis=0)    centroide = centroide / (np.linalg.norm(centroide) + 1e-12)    X_norm = np.linalg.norm(X, axis=1, keepdims=True) + 1e-12    scores = (X @ centroide) / X_norm.ravel()    s = pd.Series(scores, index=ids, name="score")    s.loc[seeds] = np.nan  # excluir los propios seeds del ranking    return s

## 5 · Modelo 2 · XGBoost con Positive-Unlabeled LearningCon solo 3 positivos reales (los seeds), planteamos el problema como *positive-unlabeled learning*:- **Positivos:** los seeds del ICP.- **Negativos sintéticos:** muestra aleatoria de restaurantes no-seed.**Aviso metodológico:** los negativos no son verdaderos negativos, sino unlabeled tratados como negativos. Este enfoque introduce ruido por definición y hace que el modelo dependa fuertemente de la semilla aleatoria de la muestra de negativos.

In [ ]:
try:    import xgboost as xgbexcept ImportError:    import subprocess    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xgboost"])    import xgboost as xgbdef scores_xgboost(X: np.ndarray, ids: pd.Series, seeds: list[str],                   n_negativos: int = 200, random_state: int = 42) -> pd.Series:    rng = np.random.default_rng(random_state)    mask_pos = ids.isin(seeds).values    idx_pos = np.where(mask_pos)[0]    idx_disponibles = np.where(~mask_pos)[0]    idx_neg = rng.choice(idx_disponibles,                         size=min(n_negativos, len(idx_disponibles)),                         replace=False)    X_train = np.vstack([X[idx_pos], X[idx_neg]])    y_train = np.concatenate([np.ones(len(idx_pos)), np.zeros(len(idx_neg))])    modelo = xgb.XGBClassifier(        n_estimators=100, max_depth=3, learning_rate=0.1,        eval_metric="logloss", use_label_encoder=False,        random_state=random_state, verbosity=0,    )    modelo.fit(X_train, y_train)    probas = modelo.predict_proba(X)[:, 1]    s = pd.Series(probas, index=ids, name="score")    s.loc[seeds] = np.nan    return s

## 6 · Baseline manual — filtro por reglas durasSimulamos lo que haría un comercial sin ML: **filtro binario** por características del ICP. Para hacerlo comparable con los modelos (que devuelven un ranking), tratamos "cumple el filtro" como score 1 y "no cumple" como score 0, y desempatamos aleatoriamente dentro del grupo que cumple.Este baseline es **el punto de referencia real**: si el modelo no supera este filtro con criterio, no aporta valor.

In [ ]:
def baseline_manual(df: pd.DataFrame, seeds: list[str], reglas: dict,                    random_state: int = 42) -> pd.Series:    """Devuelve un score binario+ruido según cumpla el filtro.    Las reglas se pasan como dict: columna -> lista de valores permitidos (o True/False)."""    mask = pd.Series(True, index=df.index)    for col, valores in reglas.items():        if col not in df.columns:            continue        if isinstance(valores, list):            mask &= df[col].isin(valores)        elif isinstance(valores, bool):            mask &= (df[col].fillna(False) == valores)    # Score = 1 si cumple, 0 si no. Añadimos ruido pequeño para desempatar.    rng = np.random.default_rng(random_state)    base = mask.astype(float).values    ruido = rng.uniform(0, 0.001, size=len(base))    scores = base + ruido    s = pd.Series(scores, index=df["restaurante_id"], name="score")    s.loc[seeds] = np.nan    return s# Definimos las reglas de filtro para cada ICPREGLAS_PREMIUM = {    "tiene_web": True,    "presencia_digital": ["activa", "sofisticada"],    "posicionamiento_precio": ["premium", "alta_gama"],    "barrio": ["JUSTICIA", "PALACIO", "CORTES", "SOL"],}REGLAS_ASIATICO = {    "cocina": ["chinese", "japanese", "asian", "thai", "vietnamese", "korean", "sushi"],}# Contamos cuántos restaurantes cumple cada filtroscores_baseline_premium = baseline_manual(gold, seeds_premium_ids, REGLAS_PREMIUM)scores_baseline_asiatico = baseline_manual(gold, seeds_asiatico_ids, REGLAS_ASIATICO)n_prem = int((scores_baseline_premium > 0.5).sum())n_asia = int((scores_baseline_asiatico > 0.5).sum())print(f"Baseline PREMIUM: {n_prem} restaurantes cumplen el filtro.")print(f"Baseline ASIÁTICO: {n_asia} restaurantes cumplen el filtro.")

## 7 · Evaluación leave-one-outPara cada ICP, apartamos uno de los seeds, entrenamos con los otros 2, y comprobamos si el seed apartado aparece en el top-K del ranking. Repetimos para cada seed.**Aviso importante sobre esta métrica:** el LOO con solo 3 seeds y un universo de 1.685 restaurantes es una prueba de estrés extrema. Con K=20, la probabilidad basal (azar) es de solo 20/1685 ≈ 1,2%. Cualquier modelo con Recall@20 significativamente > 0,012 estaría aportando señal. Un Recall@20 = 0 en las 3 iteraciones no descarta que el modelo devuelva rankings útiles cualitativamente — solo indica que no coloca el seed apartado en las primeras 20 posiciones cuando entrena con 2 puntos en un espacio de ~150 dimensiones.

In [ ]:
def evaluar_loo(df: pd.DataFrame, X: np.ndarray, ids: pd.Series,                seeds: list[str], reglas_baseline: dict, K: int = 20) -> pd.DataFrame:    resultados = []    for seed_out in seeds:        seeds_in = [s for s in seeds if s != seed_out]        # Modelo 1 · Similitud coseno        s1 = scores_coseno(X, ids, seeds_in)        s1.loc[seeds_in] = np.nan        rank1 = s1.dropna().sort_values(ascending=False)        pos_m1 = list(rank1.index).index(seed_out) + 1 if seed_out in rank1.index else -1        in_topk_m1 = 0 < pos_m1 <= K        # Modelo 2 · XGBoost        s2 = scores_xgboost(X, ids, seeds_in)        s2.loc[seeds_in] = np.nan        rank2 = s2.dropna().sort_values(ascending=False)        pos_m2 = list(rank2.index).index(seed_out) + 1 if seed_out in rank2.index else -1        in_topk_m2 = 0 < pos_m2 <= K        # Baseline manual        s3 = baseline_manual(df, seeds_in, reglas_baseline)        s3.loc[seeds_in] = np.nan        rank3 = s3.dropna().sort_values(ascending=False)        pos_m3 = list(rank3.index).index(seed_out) + 1 if seed_out in rank3.index else -1        in_topk_m3 = 0 < pos_m3 <= K        universo = len(rank1)        nombre = df.loc[df["restaurante_id"] == seed_out, "nombre"].values[0]        resultados.append({            "seed_apartado": nombre,            "pos_baseline": pos_m3,            "topK_baseline": in_topk_m3,            "pos_coseno": pos_m1,            "topK_coseno": in_topk_m1,            "pos_xgboost": pos_m2,            "topK_xgboost": in_topk_m2,            "P_azar": K / universo,            "universo": universo,        })    return pd.DataFrame(resultados)

## 8 · Evaluación ICP premium

In [ ]:
K = 20eval_premium = evaluar_loo(gold, X, ids, seeds_premium_ids, REGLAS_PREMIUM, K=K)print(f"=== LOO ICP PREMIUM (K={K}) ===\n")print(eval_premium.to_string(index=False))

## 9 · Evaluación ICP asiático

In [ ]:
eval_asiatico = evaluar_loo(gold, X, ids, seeds_asiatico_ids, REGLAS_ASIATICO, K=K)print(f"=== LOO ICP ASIÁTICO (K={K}) ===\n")print(eval_asiatico.to_string(index=False))

## 10 · Resumen comparativoConsolidamos las métricas: Recall@K por modelo y por ICP, posición media del seed recuperado, y lift sobre el azar.

In [ ]:
def resumen(eval_df: pd.DataFrame, K: int, nombre_icp: str) -> pd.DataFrame:    p_azar = eval_df["P_azar"].mean()    filas = [        {"ICP": nombre_icp, "Modelo": "Baseline manual",         f"Recall@{K}": eval_df["topK_baseline"].mean(),         "Pos. media": eval_df["pos_baseline"].mean(),         "Lift vs azar": eval_df["topK_baseline"].mean() / p_azar if p_azar > 0 else float("nan")},        {"ICP": nombre_icp, "Modelo": "Similitud coseno",         f"Recall@{K}": eval_df["topK_coseno"].mean(),         "Pos. media": eval_df["pos_coseno"].mean(),         "Lift vs azar": eval_df["topK_coseno"].mean() / p_azar if p_azar > 0 else float("nan")},        {"ICP": nombre_icp, "Modelo": "XGBoost (PU)",         f"Recall@{K}": eval_df["topK_xgboost"].mean(),         "Pos. media": eval_df["pos_xgboost"].mean(),         "Lift vs azar": eval_df["topK_xgboost"].mean() / p_azar if p_azar > 0 else float("nan")},    ]    return pd.DataFrame(filas)tabla = pd.concat([    resumen(eval_premium, K, "PREMIUM"),    resumen(eval_asiatico, K, "ASIÁTICO"),], ignore_index=True)print(f"=== RESUMEN COMPARATIVO (K={K}) ===\n")print(tabla.to_string(index=False))# Guardar métricasruta_metricas = ROOT / "data" / "processed" / "metricas_modelos.csv"ruta_metricas.parent.mkdir(parents=True, exist_ok=True)tabla.to_csv(ruta_metricas, index=False)print(f"\nMétricas guardadas en: {ruta_metricas}")

## 11 · Visualización — posición media del seed apartadoEn vez de graficar Recall@K (que probablemente sea 0 en varios casos), visualizamos la **posición media del seed apartado en el ranking**. Un número más bajo indica que el modelo lo coloca más cerca del top. La línea horizontal marca la posición media esperada por azar (~843 en un universo de ~1.685).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))modelos = ["Baseline\nmanual", "Similitud\ncoseno", "XGBoost\n(PU)"]colores = ["#94a3b8", "#3b82f6", "#f59e0b"]azar_pos = eval_premium["universo"].mean() / 2# --- Premium ---pos_prem = [eval_premium["pos_baseline"].mean(),            eval_premium["pos_coseno"].mean(),            eval_premium["pos_xgboost"].mean()]bars1 = ax[0].bar(modelos, pos_prem, color=colores, edgecolor="black", linewidth=0.5)ax[0].axhline(azar_pos, color="grey", linestyle="--", alpha=0.5, label=f"Azar ≈ {azar_pos:.0f}")ax[0].set_ylabel("Posición media del seed apartado", fontsize=11)ax[0].set_title("ICP PREMIUM", fontsize=12)ax[0].invert_yaxis()  # menor posición = mejorfor bar, val in zip(bars1, pos_prem):    ax[0].text(bar.get_x() + bar.get_width() / 2, val,               f"{val:.0f}", ha="center", va="bottom", fontsize=10, fontweight="bold")ax[0].legend()# --- Asiático ---pos_asia = [eval_asiatico["pos_baseline"].mean(),            eval_asiatico["pos_coseno"].mean(),            eval_asiatico["pos_xgboost"].mean()]bars2 = ax[1].bar(modelos, pos_asia, color=colores, edgecolor="black", linewidth=0.5)ax[1].axhline(azar_pos, color="grey", linestyle="--", alpha=0.5, label=f"Azar ≈ {azar_pos:.0f}")ax[1].set_ylabel("Posición media del seed apartado", fontsize=11)ax[1].set_title("ICP ASIÁTICO", fontsize=12)ax[1].invert_yaxis()for bar, val in zip(bars2, pos_asia):    ax[1].text(bar.get_x() + bar.get_width() / 2, val,               f"{val:.0f}", ha="center", va="bottom", fontsize=10, fontweight="bold")ax[1].legend()plt.suptitle("Posición media del seed apartado en el ranking (LOO, K={})".format(K),             fontsize=13, fontweight="bold", y=1.02)plt.tight_layout()salida = ROOT / "data" / "processed" / "comparacion_modelos.png"salida.parent.mkdir(parents=True, exist_ok=True)plt.savefig(salida, dpi=120, bbox_inches="tight")plt.show()print(f"\nGráfico guardado en: {salida}")

## 12 · Conclusiones honestas### Lectura de los resultados**En Recall@20 estricto**, ninguno de los tres enfoques (baseline manual, similitud coseno, XGBoost) supera al azar de forma consistente en las 3 iteraciones LOO de cada ICP. Con K=20 sobre un universo de ~1.685 restaurantes, la probabilidad basal es de 1,2% — muy exigente cuando entrenamos con solo 2 puntos por iteración.**En posición media del seed apartado**, sí observamos diferencias interpretables entre modelos y entre ICPs:- La **similitud coseno** con el ICP asiático (seeds más homogéneos entre sí) coloca al seed apartado significativamente por encima de la línea del azar. Con el ICP premium (seeds más heterogéneos), no supera el azar.- **XGBoost** presenta comportamiento inestable: la aleatoriedad de la muestra de negativos sintéticos hace que sus resultados varíen entre ICPs sin patrón claro.- El **baseline manual** funciona como referencia: se comporta parecido al azar dentro de su conjunto de "positivos" porque desempata al azar.### Interpretación honesta**Estos resultados NO permiten afirmar que el sistema aprende el ICP mejor que el azar bajo la métrica LOO@20.** Es una limitación real que hay que reconocer.Sin embargo, tampoco descartan que el sistema aporte valor cualitativo. En la app Streamlit, cuando se cargan los 3 seeds completos (no en modo LOO), el ranking devuelve leads cualitativamente coherentes con el ICP — restaurantes premium reconocibles en el caso de Demo 1, restaurantes asiáticos variados en Demo 2. **La coherencia cualitativa del ranking no queda capturada por la métrica LOO@20**, que es una prueba de estrés matemática, no una prueba de utilidad comercial.### Consecuencia metodológica**Con esta evidencia, el sistema debe presentarse como una prueba de concepto cualitativa**, no como un recomendador validado cuantitativamente. La validación cuantitativa robusta requeriría:- Un conjunto de seeds más amplio (10+ ejemplos por ICP).- Idealmente, datos históricos reales de conversión para validación supervisada.- O bien, cambiar la métrica: en vez de recuperar 1 seed en el top-20 estricto, medir consistencia del ranking entre diferentes composiciones de seeds del mismo ICP.### Decisión sobre el modelo en producciónSe mantiene la **similitud coseno** como modelo integrado en la app por tres motivos que sí quedan respaldados empíricamente:1. Es **determinístico**: mismo input, mismo output (a diferencia de XGBoost que depende de la semilla de negativos sintéticos).2. Es **más estable** entre ICPs distintos.3. Es **directamente interpretable**: cada score se puede explicar por la contribución de las features al centroide (aunque esta explicabilidad aún no está expuesta al usuario en la app — línea futura).### Líneas futuras- **Incorporar embedding del resumen_llm** como parte del vector — actualmente el modelo solo usa features estructuradas.- **Validación con conjuntos de seeds más amplios** (5-10 seeds por ICP).- **Añadir descomposición del score al usuario** en la app (mostrar las features que más contribuyen).- **Ampliar el enriquecimiento del INE** (población, edad media, tamaño del hogar) para otros verticales B2B.